# DWS-Bench: Qwen 0.5B No-CoT

Runs three independent budgets (128, 256, 512). Each notebook is designed to fit a 12-hour Kaggle session.

In [ ]:
%pip install -q bitsandbytes pandas

from pathlib import Path
import json, os, shutil, subprocess, sys, time
import pandas as pd

WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'StateMachine'
INPUT_ROOT = Path('/kaggle/input')
REPO_URL = ''

if REPO_URL:
    if PROJECT_DIR.exists(): shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    candidates = [p for p in INPUT_ROOT.rglob('run_eval.py') if (p.parent / 'generate_all.py').exists() and (p.parent / 'data').exists()]
    if not candidates: raise FileNotFoundError('Upload the repository as a Kaggle Dataset or set REPO_URL.')
    source_dir = candidates[0].parent
    if PROJECT_DIR.exists(): shutil.rmtree(PROJECT_DIR)
    shutil.copytree(source_dir, PROJECT_DIR, ignore=shutil.ignore_patterns('.git','__pycache__','*.pyc','results'))
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(PROJECT_DIR)

In [ ]:
MODEL = 'qwen2.5-0.5b'
USE_COT = False
TOKEN_BUDGETS = [128, 256, 512]
DATASET = 'full'
PRECISION = '4bit'
BATCH_SIZE = 1
DEVICE = 'auto'
OUTPUT_ROOT = WORK_ROOT / 'dws_qwen05_no_cot'

dataset_path = PROJECT_DIR / 'data' / 'full_benchmark.jsonl'
if not dataset_path.exists(): raise FileNotFoundError(f'Missing benchmark: {dataset_path}')
record_count = sum(1 for line in dataset_path.open(encoding='utf-8') if line.strip())
print('Using:', dataset_path, 'records:', record_count)
if not 1000 <= record_count <= 1200: raise ValueError(f'Unexpected benchmark size: {record_count}')

In [ ]:
run_status = []
for tokens in TOKEN_BUDGETS:
    condition = f'{MODEL}_no_cot_{tokens}'
    output_dir = OUTPUT_ROOT / condition
    command = [sys.executable, 'run_eval.py', '--model', MODEL, '--dataset', DATASET, '--device', DEVICE, '--precision', PRECISION, '--batch-size', str(BATCH_SIZE), '--max-new-tokens', str(tokens), '--output-dir', str(output_dir)]
    print(f'Running {condition}')
    started = time.time()
    result = subprocess.run(command, cwd=PROJECT_DIR, env=os.environ.copy())
    run_status.append({'condition': condition, 'return_code': result.returncode, 'minutes': round((time.time()-started)/60, 2)})
pd.DataFrame(run_status).to_csv(OUTPUT_ROOT / 'run_status.csv', index=False)
display(pd.DataFrame(run_status))

In [ ]:
rows = []
for tokens in TOKEN_BUDGETS:
    condition = f'{MODEL}_no_cot_{tokens}'
    metrics_path = OUTPUT_ROOT / condition / MODEL / 'full_benchmark_metrics.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        rows.append({'model': MODEL, 'cot': False, 'max_new_tokens': tokens, 'accuracy': metrics['overall_accuracy'], 'stepwise_accuracy': metrics.get('stepwise_accuracy'), 'instances': metrics['total_instances'], 'runtime_minutes': metrics['elapsed_seconds']/60})
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT_ROOT / 'corrected_condition_summary.csv', index=False)
display(summary)
print('Saved results to', OUTPUT_ROOT)

In [ ]:
# Archive all No-CoT 0.5B results for download.
archive_path = shutil.make_archive(
    str(WORK_ROOT / "dws_qwen05_no_cot"),
    "zip",
    root_dir=WORK_ROOT,
    base_dir="dws_qwen05_no_cot",
)
print(f"Created archive: {archive_path}")